In [ ]:
from pathlib import Path
import pandas as pd
import requests
import time

PROJECT_ROOT = Path("..")

USGS_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "usgs"
USGS_RAW_DIR.mkdir(parents=True, exist_ok=True)

USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

START_YEAR = 2010
END_YEAR = 2025
MIN_MAGNITUDE = 4.0

USGS raw directory: ..\data\raw\usgs


In [ ]:
def download_usgs_year(year, min_magnitude=4.0):

    start_date = f"{year}-01-01"
    end_date = f"{year + 1}-01-01"

    output_file = USGS_RAW_DIR / f"usgs_{year}.csv"

    params = {
        "format": "geojson",
        "starttime": start_date,
        "endtime": end_date,
        "minmagnitude": min_magnitude,
        "orderby": "time-asc",
        "limit": 20000
    }

    print(f"\nDownloading {year}...")

    response = requests.get(
        USGS_URL,
        params=params,
        timeout=120
    )

    response.raise_for_status()

    data = response.json()

    records = []

    for event in data["features"]:

        properties = event["properties"]
        geometry = event["geometry"]

        if geometry is None or geometry.get("coordinates") is None:
            continue

        coordinates = geometry["coordinates"]

        if len(coordinates) < 3:
            continue

        records.append({
            "event_id": event["id"],
            "time": pd.to_datetime(
                properties["time"],
                unit="ms",
                utc=True
            ),
            "longitude": coordinates[0],
            "latitude": coordinates[1],
            "depth_km": coordinates[2],
            "magnitude": properties["mag"],
            "magnitude_type": properties["magType"],
            "place": properties["place"],
            "event_type": properties["type"],
            "status": properties["status"],
        })

    df = pd.DataFrame(records)

    df.to_csv(output_file, index=False)

    print(f"Events: {len(df):,}")
    print(f"Saved: {output_file}")

    return df

Status: 200
URL: https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2024-01-01&endtime=2024-01-08&minmagnitude=4.0&orderby=time-asc&limit=20000


In [ ]:
df_2010 = download_usgs_year(2010)

Number of events: 359


In [ ]:
print(df_2010.shape)
print(df_2010.head())
print(df_2010.isna().sum())
print(
    "Duplicate event IDs:",
    df_2010["event_id"].duplicated().sum()
)

NameError: name 'df_2010' is not defined

In [5]:
records = []

for event in data["features"]:
    properties = event["properties"]
    geometry = event["geometry"]

    records.append({
        "event_id": event["id"],
        "time": pd.to_datetime(properties["time"], unit="ms", utc=True),
        "longitude": geometry["coordinates"][0],
        "latitude": geometry["coordinates"][1],
        "depth_km": geometry["coordinates"][2],
        "magnitude": properties["mag"],
        "magnitude_type": properties["magType"],
        "place": properties["place"],
        "event_type": properties["type"],
        "status": properties["status"],
    })

usgs_test = pd.DataFrame(records)

usgs_test.head()

,event_id,time,longitude,latitude,depth_km,magnitude,magnitude_type,place,event_type,status
0,us6000m0w6,2024-01-01 00:37:09.096000+00:00,145.4596,17.6625,384.941,4.7,mb,"272 km N of Saipan, Northern Mariana Islands",earthquake,reviewed
1,us6000m357,2024-01-01 00:54:44.947000+00:00,-25.8967,-57.3045,55.881,4.4,mb,South Sandwich Islands region,earthquake,reviewed
2,us6000m0wb,2024-01-01 00:59:19.656000+00:00,107.1224,-6.8599,178.051,4.1,mb,"4 km SSW of Cianjur, Indonesia",earthquake,reviewed
3,us6000m0wa,2024-01-01 01:05:25+00:00,-70.5175,-25.3436,35.000,4.3,mb,"7 km NNW of Taltal, Chile",earthquake,reviewed
4,us6000m0wc,2024-01-01 01:43:14.869000+00:00,179.8704,-24.4120,513.317,4.6,mb,south of the Fiji Islands,earthquake,reviewed


In [ ]:
df_2010 = download_usgs_year(2010)

Shape: (359, 10)

Missing values:
event_id          0
time              0
longitude         0
latitude          0
depth_km          0
magnitude         0
magnitude_type    0
place             0
event_type        0
status            0
dtype: int64

Data types:
event_id                       object
time              datetime64[ns, UTC]
longitude                     float64
latitude                      float64
depth_km                      float64
magnitude                     float64
magnitude_type                 object
place                          object
event_type                     object
status                         object
dtype: object


In [7]:
print("Magnitude range:")
print(usgs_test["magnitude"].min(),
      usgs_test["magnitude"].max())

print("\nDepth range:")
print(usgs_test["depth_km"].min(),
      usgs_test["depth_km"].max())

print("\nLatitude range:")
print(usgs_test["latitude"].min(),
      usgs_test["latitude"].max())

print("\nLongitude range:")
print(usgs_test["longitude"].min(),
      usgs_test["longitude"].max())

print("\nMagnitude types:")
print(usgs_test["magnitude_type"].value_counts())

Magnitude range:
4.0 7.5

Depth range:
4.785 659.944

Latitude range:
-62.3294 76.065

Longitude range:
-179.8743 179.8704

Magnitude types:
magnitude_type
mb     312
mww     27
mwr     16
mw       2
ml       1
mwb      1
Name: count, dtype: int64


In [8]:
print("\nEvent types:")
print(usgs_test["event_type"].value_counts())


Event types:
event_type
earthquake    359
Name: count, dtype: int64
